### 被clip的token优势估计已经是一个常数，因此梯度直接置0

### clip的使用要配合“悲观下界”一起理解（两者是一体的）

clip + “悲观下界”两者工条效果对于正负优势是不对称的：
- 对于正优势，限制过大幅度的参数更新，这个容易理解，防止步子迈得过大。
- 对于负优势，限制过小幅度的参数更新（负优势样本对token logits的影响）
   - 总结下来就是：频繁的、绝对值低的负优势会不仅降低当前已经很小的token logits造成崩溃，也会频繁提升其他所有token的logits造成不稳定

### 详细解释
- ppo-clip(GRPO也是参考这个策略)的“clip + min悲观下界”策略怎么理解
- 被clip的token梯度计算为什么是0，以及验证



#### clip
- clip裁剪当值策略更新幅度过大，若不对r = π_theta / π_old 进行限制，当策略变化剧烈时，优化过程可能不稳定


#### min 悲观下界：选择更保守的估计值
通过min(rA,clip(rA),目标函数始终选择<b>更保守（更低）的估计值</b>

这一设计保证了：
- 优势为正（A>0），不鼓励过度优化（超过一定幅度即被裁剪，梯度贡献为0）
- 优势为负（A<0），避免过于敏感的乘法（当前动作概率很小，则不需要再降低了，否则可能会崩溃）

### 为什么被clip的token梯度为0

In [1]:
import torch

# 1. 创建数据： 0.5在clip区间内，2.0在clip区间外
x = torch.tensor([0.5,2.0],requires_grad=True)

# 2.阶段[0,1]
y = torch.clamp(x,0,1)

# 3.反向传播
y.sum().backward()

# 打印
print(f"原始数值：{x.data}")
print(f"梯度结果：{x.grad}")

原始数值：tensor([0.5000, 2.0000])
梯度结果：tensor([1., 0.])


可以发现被clip的2.0的梯度为0